# 🌸 Émo — Backend
Lance les cellules dans l'ordre. À la fin tu obtiens une **URL publique** à coller dans le frontend.

> ⚠️ Va dans **Exécution → Modifier le type d'exécution → GPU T4** avant de commencer.

In [ ]:
# ── Cellule 1 : Installation des dépendances ──────────────────────
!pip install -q fastapi uvicorn pyngrok transformers accelerate sentencepiece torch nest_asyncio

In [ ]:
# ── Cellule 2 : Imports & vérification GPU ───────────────────────
import torch
print('🖥️  Device:', 'CUDA ✅' if torch.cuda.is_available() else 'CPU ⚠️ (lent)')
if torch.cuda.is_available():
    print('   GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Cellule 3 : Token ngrok ──────────────────────────────────────
# Crée un compte GRATUIT sur https://ngrok.com → Dashboard → Your Authtoken
NGROK_TOKEN = ''  # ← colle ton token ici

# Token Hugging Face (https://huggingface.co/settings/tokens)
HF_TOKEN = ''  # ← colle ton token HF ici (Read)

from huggingface_hub import login
if HF_TOKEN:
    login(HF_TOKEN)
    print('✅ HuggingFace connecté')
else:
    print('⚠️  Pas de HF token — le modèle sera peut-être inaccessible')

In [ ]:
# ── Cellule 4 : Chargement du modèle Qwen ───────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
import re, json
from threading import Thread

MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'📦 Chargement de {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print('✅ Modèle prêt !')

In [ ]:
# ── Cellule 5 : API FastAPI + ngrok ─────────────────────────────
import nest_asyncio, uvicorn, asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import List, Optional
from pyngrok import ngrok, conf

nest_asyncio.apply()

# ── Emotion detection ────────────────────────────────────────────
EMOTION_PATTERNS = {
    'joy':      r'\b(haha|hehe|😂|😄|happy|great|awesome|love|wonderful|yay|congrats|amazing|excellent|glad|excited|joyful|lol|xd|super|génial|content|heureux|trop bien)\b',
    'sadness':  r'\b(sad|cry|crying|😢|😭|sorry|miss|lost|alone|broken|depressed|triste|désolé|malheureux|pleurs|déprimé)\b',
    'anger':    r'\b(angry|furious|mad|hate|wtf|damn|stupid|rage|annoyed|frustrated|énervé|colère|nul|horrible|furieux)\b',
    'fear':     r'\b(scared|afraid|worried|nervous|anxious|panic|terror|peur|stressé|inquiet|angoisse|effrayé)\b',
    'surprise': r'\b(wow|whoa|omg|really|seriously|shocked|surprised|incroyable|vraiment|attends|quoi|sérieux)\b',
    'disgust':  r'\b(gross|disgusting|eww|yuck|awful|horrible|nasty|dégueulasse|beurk|horrible)\b',
}
EMOTION_META = {
    'joy':      {'color':'#FFD166','label':'Joyeux',    'eyes':'happy'},
    'sadness':  {'color':'#60a5fa','label':'Triste',    'eyes':'sad'},
    'anger':    {'color':'#f87171','label':'En colère', 'eyes':'angry'},
    'fear':     {'color':'#a78bfa','label':'Apeuré',    'eyes':'scared'},
    'surprise': {'color':'#34d399','label':'Surpris',   'eyes':'surprised'},
    'disgust':  {'color':'#a3a300','label':'Dégoûté',   'eyes':'disgust'},
    'neutral':  {'color':'#94a3b8','label':'Neutre',    'eyes':'neutral'},
}

def detect_emotion(text):
    t = text.lower()
    for emotion, pattern in EMOTION_PATTERNS.items():
        if re.search(pattern, t, re.IGNORECASE):
            return {'emotion': emotion, **EMOTION_META[emotion]}
    return {'emotion': 'neutral', **EMOTION_META['neutral']}

SYSTEM_PROMPT = """Tu es Émo, une IA amicale, empathique et curieuse.
Tu réponds toujours en français sauf si on te parle dans une autre langue.
Tu es expressif·ve, chaleureux·se, et tu adaptes ton ton à l'émotion de la conversation.
Tu es là pour aider, discuter, coder, et soutenir les gens."""

# ── FastAPI app ───────────────────────────────────────────────────
app = FastAPI(title='Émo API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    messages: List[Message]
    stream: Optional[bool] = True

class EmotionRequest(BaseModel):
    text: str

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/emotion')
def emotion_route(req: EmotionRequest):
    return detect_emotion(req.text)

@app.post('/chat')
def chat(req: ChatRequest):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for m in req.messages:
        msgs.append({'role': m.role, 'content': m.content})

    text_input = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_input, return_tensors='pt').to(DEVICE)

    if req.stream:
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        gen_kwargs = dict(**inputs, streamer=streamer, max_new_tokens=512,
                          do_sample=True, temperature=0.7, top_p=0.9,
                          pad_token_id=tokenizer.eos_token_id)

        def generate_and_stream():
            thread = Thread(target=model.generate, kwargs=gen_kwargs)
            thread.start()
            full = ''
            for token in streamer:
                full += token
                yield f"data: {json.dumps({'token': token, 'done': False})}\n\n"
            em = detect_emotion(full)
            yield f"data: {json.dumps({'token': '', 'done': True, 'emotion': em})}\n\n"

        return StreamingResponse(generate_and_stream(), media_type='text/event-stream')
    else:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=512, do_sample=True,
                                  temperature=0.7, top_p=0.9,
                                  pad_token_id=tokenizer.eos_token_id)
        generated = out[0][inputs['input_ids'].shape[1]:]
        result = tokenizer.decode(generated, skip_special_tokens=True)
        return {'response': result, 'emotion': detect_emotion(result)}

# ── ngrok tunnel ─────────────────────────────────────────────────
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(8000)
print('\n' + '='*60)
print(f'🌸 Émo API en ligne : {public_url}')
print('='*60)
print('👆 Copie cette URL et colle-la dans le frontend sur HuggingFace')
print('   (champ API URL en haut à droite de l\'interface)\n')

uvicorn.run(app, host='0.0.0.0', port=8000)